# 07 · Layer 3：Sequential、Parallel、Loop

Layer 3 是多 agent。而多 agent 的第一個問題是：**誰決定執行順序？**

ADK 給了兩種答案：

| | 誰決定順序 | 可預期性 | 本教材位置 |
|---|---|---|---|
| **Workflow Agent** | **你**，寫死在程式裡 | 100% 可預期 | 本章 + 第 08 章 |
| **LLM 自主委派** | **模型** | 每次都可能不同 | 第 09 章 |

本章講三個內建的 Workflow Agent。它們的共同點是：**不需要 LLM 決定流程**，
所以不會失控、不會多花 token 在「決定要做什麼」上面。

## 0. 環境

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

> **版本提醒**：`SequentialAgent` / `ParallelAgent` / `LoopAgent` 從
> ADK Python 2.8.0 起被標記為 **deprecated**，官方方向是改用 2.0 的
> `Workflow` 圖形化引擎（第 10 章）。
>
> 但它們仍然可用，而且是理解編排的最好起點——概念比 `Workflow` 單純，
> 也是跨語言（Go / TypeScript）都有的共同原語。**先懂這三個，
> 再看 `Workflow` 會輕鬆很多。**

## 1. `SequentialAgent`：一個接一個

最單純的模式：sub_agents 依序執行，前一個的 `output_key` 寫進 state，
後一個用 `{key?}` 讀出來。

```
使用者輸入 → agent1 → state["a"] → agent2 → state["b"] → agent3 → 輸出
```

情境：客訴處理。分類 → 擬回覆 → 潤稿。

In [2]:
from google.adk.agents import LlmAgent, SequentialAgent

classifier = LlmAgent(
    name="classifier",
    model=get_model(),
    instruction=(
        "你是客訴分類員。判斷使用者的抱怨屬於哪一類："
        "「物流」「品質」「客服態度」「價格」其中之一。"
        "只回類別兩到四個字，不要其他內容。"
    ),
    output_key="category",
)

drafter = LlmAgent(
    name="drafter",
    model=get_model(),
    instruction=(
        "你是客服。這是一則「{category?}」類的客訴。\n"
        "針對使用者的抱怨寫一段道歉與處理方式，三句話以內，繁體中文。"
    ),
    output_key="draft",
)

polisher = LlmAgent(
    name="polisher",
    model=get_model(),
    instruction=(
        "把下面這段客服回覆潤飾得更誠懇、更專業，保持三句話以內，只回潤飾後的內容：\n\n"
        "{draft?}"
    ),
    output_key="final_reply",
)

complaint_pipeline = SequentialAgent(
    name="complaint_pipeline",
    sub_agents=[classifier, drafter, polisher],
)

print("pipeline 內的順序:", [a.name for a in complaint_pipeline.sub_agents])

pipeline 內的順序: ['classifier', 'drafter', 'polisher']


In [3]:
from google.adk.runners import InMemoryRunner

runner = InMemoryRunner(agent=complaint_pipeline, app_name="concept_track")
sid = await new_session(runner)

complaint = "我上週訂的東西到現在還沒收到，查物流也沒更新，打客服電話都沒人接。"
await ask(runner, complaint, session_id=sid, trace=True)

print("\n--- 每一站留在 state 的東西 ---")
print_state(await peek_state(runner, sid))

  💬 [classifier] 物流


  💬 [drafter] 非常抱歉讓您久等了，我們深感抱歉未能及時送達並造成您的困擾。我們已立即為您聯繫物流公司追蹤包裹最新動態，並會盡快由專人與您回報處理進度。


  💬 [polisher] 對於電話未能及時接聽與包裹延誤，我們致上最深的歉意，造成您的困擾我們感同身受。我們已緊急聯繫物流公司全面追蹤您的包裹動態，一有最新消息將由專人第一時間向您回報，請您稍候。

--- 每一站留在 state 的東西 ---
  category: 物流
  draft: 非常抱歉讓您久等了，我們深感抱歉未能及時送達並造成您的困擾。我們已立即為您聯繫物流公司追蹤包裹最新動態，並會盡快由專人與您回報處理進度。
  final_reply: 對於電話未能及時接聽與包裹延誤，我們致上最深的歉意，造成您的困擾我們感同身受。我們已緊急聯繫物流公司全面追蹤您的包裹動態，一有最新消息將由專人第一時間向您回報，請您稍候。


三個 agent 依序跑完，每一站的產出都留在 state 裡。這代表：

- **可以中途檢查**——出錯時你知道是哪一站壞的
- **可以重跑單一站**——不用整條重來

這是 Orchestration 相對於「一個大 prompt 做完所有事」的最大好處。

## 2. `ParallelAgent`：同時跑，互不相通

當幾個子任務**彼此沒有依賴**時，依序跑就是在浪費時間。

```
             ┌─→ 正方觀點 ─┐
使用者輸入 ──┼─→ 反方觀點 ─┼─→ 各自寫進 state
             └─→ 成本分析 ─┘
```

**關鍵限制**：平行的分支之間**看不到彼此的 state**。
它們同時起跑，誰也讀不到誰的結果。

In [4]:
from google.adk.agents import ParallelAgent


def make_reviewers() -> list[LlmAgent]:
    """每次呼叫都給一組**全新**的 agent 實例。

    ADK 規定「一個 agent 只能有一個父節點」，所以同一批 agent 沒辦法同時掛在
    ParallelAgent 和 SequentialAgent 底下。用工廠函式產生新實例是標準做法。
    """
    return [
        LlmAgent(
            name="pros_agent",
            model=get_model(),
            instruction="你是樂觀派分析師。針對使用者提出的方案，列出三個優點，條列式，繁體中文。",
            output_key="pros",
        ),
        LlmAgent(
            name="cons_agent",
            model=get_model(),
            instruction="你是風險控管。針對使用者提出的方案，列出三個風險，條列式，繁體中文。",
            output_key="cons",
        ),
        LlmAgent(
            name="cost_agent",
            model=get_model(),
            instruction="你是財務。針對使用者提出的方案，估算主要成本項目，三點以內，繁體中文。",
            output_key="cost",
        ),
    ]


board = ParallelAgent(name="board", sub_agents=make_reviewers())

In [5]:
import time

topic = "我們公司要不要把所有內部工具都改成 AI Agent 驅動？"

runner_p = InMemoryRunner(agent=board, app_name="concept_track")
sid_p = await new_session(runner_p)

start = time.perf_counter()
await ask(runner_p, topic, session_id=sid_p)
parallel_secs = time.perf_counter() - start

print(f"平行執行耗時: {parallel_secs:.1f} 秒\n")
print_state(await peek_state(runner_p, sid_p), max_len=80)

平行執行耗時: 6.5 秒

  cost: 評估全面導入 AI Agent 的主要成本如下：  1. **API 呼叫與基礎設施費用**：大量使用 LLM 進行推理會產生高昂的 Token 消耗成本，且需 …（共 223 字）
  cons: 我是風險控管。針對將所有內部工具全面改為 AI Agent 驅動的方案，評估出以下三個主要風險：  *   **資安與機密外洩風險**：全面導入 AI Agen …（共 423 字）
  pros: 這是一個極具前瞻性且令人興奮的想法！將所有內部工具改為 AI Agent 驅動，絕對能為公司帶來突破性的轉變。以下是三個主要的優點：  1. **極致的生產力與 …（共 479 字）


### 對照：同樣三件事改成依序做

> **⚠️ 一個 agent 只能有一個父節點。**
>
> 想把同一批 agent 換個編排方式跑，不能直接重用實例——會得到
> `Agent 'pros_agent' already has a parent agent, current parent: 'board'`。
> 所以下面用 `make_reviewers()` 產生**全新的一組**。
> 這也是為什麼實務上 agent 常常寫成工廠函式而不是模組層級的變數。

In [6]:
serial_board = SequentialAgent(name="serial_board", sub_agents=make_reviewers())
runner_s = InMemoryRunner(agent=serial_board, app_name="concept_track")
sid_s = await new_session(runner_s)

start = time.perf_counter()
await ask(runner_s, topic, session_id=sid_s)
serial_secs = time.perf_counter() - start

print(f"平行: {parallel_secs:.1f} 秒")
print(f"依序: {serial_secs:.1f} 秒")

平行: 6.5 秒
依序: 10.7 秒


> 註：本教材在 `shared/runtime.py` 有一道 12 RPM 的全域節流閘門，
> 會壓縮平行的優勢。實際差距在沒有節流時更明顯。

### 平行完之後要有人收尾

`ParallelAgent` 只負責「同時跑完」，它不會幫你整合。
標準做法是外面再包一層 `SequentialAgent`：**先平行、再匯總**。

In [7]:
synthesizer = LlmAgent(
    name="synthesizer",
    model=get_model(),
    instruction=(
        "你是決策顧問。根據以下三份分析，給出一個明確建議（做 / 不做 / 有條件做）"
        "並說明理由，五句話以內，繁體中文。\n\n"
        "【優點】\n{pros?}\n\n【風險】\n{cons?}\n\n【成本】\n{cost?}"
    ),
    output_key="decision",
)

full_board = SequentialAgent(
    name="full_board",
    # 同樣的理由：board 已經是別人的父節點了，這裡要一個新的
    sub_agents=[ParallelAgent(name="board2", sub_agents=make_reviewers()), synthesizer],
)

runner_f = InMemoryRunner(agent=full_board, app_name="concept_track")
sid_f = await new_session(runner_f)
await ask(runner_f, topic, session_id=sid_f)

state = await peek_state(runner_f, sid_f)
print("=== 最終建議 ===")
print(state.get("decision"))

=== 最終建議 ===
建議採用「**有條件做**」的策略。雖然 AI Agent 能大幅提升生產力與效率，但伴隨著高昂的基礎設施成本、資安漏洞及幻覺誤判等重大風險。建議公司應採分階段導入，優先選定低風險的非核心任務進行試點。同時，必須建立嚴格的資安權限控管與人工覆核機制，以確保營運穩定並防範資料外洩。


這是最常用的多 agent 骨架：**fan-out（平行展開）→ join（匯總）**。

## 3. `LoopAgent`：重複到滿意為止

```
  ┌──────────────────────────┐
  │  generator → critic      │  ← 重複
  └────────┬─────────────────┘
           │ critic 呼叫 exit_loop 才會停
           ▼
```

**最重要的一件事：`LoopAgent` 自己不知道什麼時候該停。**
停止是 sub_agent 的責任——某個 agent 要呼叫 `exit_loop` 工具。
忘了這件事，它就會一路跑到 `max_iterations` 為止（然後帳單來了）。

In [8]:
from google.adk.agents import LoopAgent
from google.adk.tools import exit_loop

generator = LlmAgent(
    name="slogan_generator",
    model=get_model(),
    instruction=(
        "你是文案。為「給工程師的保溫杯」寫一句廣告標語。\n"
        "如果下面有前一版和評語，請根據評語改寫；沒有的話就寫第一版。\n"
        "只回標語本身，不要解釋。\n\n"
        "前一版：{slogan?}\n評語：{critique?}"
    ),
    output_key="slogan",
)

critic = LlmAgent(
    name="slogan_critic",
    model=get_model(),
    instruction=(
        "你是嚴格的創意總監。評價這句標語：{slogan?}\n\n"
        "如果它同時做到「16 字以內」「有具體畫面」「不用『科技』『創新』這類空話」，"
        "就**呼叫 exit_loop 工具**結束流程。\n"
        "否則用一句話指出最該改的地方，不要呼叫工具。"
    ),
    tools=[exit_loop],
    output_key="critique",
)

refine_loop = LoopAgent(
    name="refine_loop",
    sub_agents=[generator, critic],
    max_iterations=4,  # 安全閥：無論如何最多四輪
)

In [9]:
runner_l = InMemoryRunner(agent=refine_loop, app_name="concept_track")
sid_l = await new_session(runner_l)

await ask(runner_l, "開始", session_id=sid_l, trace=True)

final = await peek_state(runner_l, sid_l)
print(f"\n=== 最終標語 ===\n{final.get('slogan')}")

  💬 [slogan_generator] 程式會當機，熱度不當機。


  🔧 [slogan_critic] 呼叫 exit_loop({})
  ↩️  [slogan_critic] exit_loop 回傳 {'result': None}

=== 最終標語 ===
程式會當機，熱度不當機。


### `max_iterations` 不是可選的

拿掉 `exit_loop` 工具、或 critic 永遠不滿意，`LoopAgent` 就會一直跑。
**`max_iterations` 是你唯一的安全閥。**

一輪 = 2 個 agent = 至少 2 次模型呼叫。`max_iterations=10` 就是 20 次呼叫。

In [10]:
runaway = LoopAgent(
    name="runaway",
    sub_agents=[
        LlmAgent(
            name="counter",
            model=get_model(),
            instruction="說一個數字，只回數字。前一個是 {n?}",
            output_key="n",
        )
    ],
    max_iterations=3,  # ← 沒有 exit_loop，全靠這行
)

runner_r = InMemoryRunner(agent=runaway, app_name="concept_track")
sid_r = await new_session(runner_r)
await ask(runner_r, "開始", session_id=sid_r, trace=True)
print("\n跑了 3 輪就被 max_iterations 擋下來——不是它自己想停的。")

  💬 [counter] 1


  💬 [counter] 2


  💬 [counter] 14

跑了 3 輪就被 max_iterations 擋下來——不是它自己想停的。


## 4. 怎麼選

| 你的狀況 | 用哪個 |
|---|---|
| 步驟固定，後面要用前面的結果 | `SequentialAgent` |
| 幾件事互不相干，想省時間 | `ParallelAgent` |
| 要反覆改進到達標 | `LoopAgent` + `exit_loop` |
| 有條件分支（if/else） | 三個都做不到 → `Workflow`（第 10 章） |
| 順序要讓模型自己決定 | → LLM 委派（第 09 章） |

**`ParallelAgent` 的分支看不到彼此的 state**，需要交流就得改用
`Workflow` 的 fan-out-join。

## 本章重點

- **Workflow Agent 的順序是你寫死的**，不需要 LLM 決定，所以可預期、也比較省。
- **`SequentialAgent`** 靠 `output_key` + `{key?}` 傳資料，
  每一站的產出都留在 state，方便除錯與重跑。
- **`ParallelAgent` 的分支互相看不到 state**。標準骨架是
  「`SequentialAgent(ParallelAgent(...), 匯總 agent)`」。
- **一個 agent 只能有一個父節點**，換編排方式要用工廠函式產生新實例。
- **`LoopAgent` 自己不知道何時該停**——停止是 sub_agent 呼叫 `exit_loop` 的責任，
  `max_iterations` 是唯一的安全閥。
- 這三個在 **ADK 2.8.0 已標記 deprecated**，但仍是理解編排最好的起點。

## 動手練習

1. 把第 1 節的 `polisher` 拿掉，改成兩站，比較輸出品質差多少。
2. 第 2 節的 `synthesizer` instruction 裡故意把 `{cost?}` 拼錯成 `{costs?}`，
   看會發生什麼（提示：問號救了你）。
3. 第 3 節把 `critic` 的 `tools=[exit_loop]` 拿掉，重跑，
   確認它一定會跑滿 4 輪。

---
**下一站 → `08_orchestration.ipynb`**：把這些組成一條真正的業務流程。